In [13]:
import numpy as np

t1 = np.array([-1.5, -0.8, 0.0, 0.9, 2.3], dtype=np.float32)

t2 = np.array([0.1, 0.5, 1.2, 2.0, 3.5], dtype=np.float32)

t3 = np.array([-3.0, -2.1, -1.4, -0.6, -0.1], dtype=np.float32)

t4 = np.array([5.0, 5.0, 5.0], dtype=np.float32)

t5 = np.array([1e-9, 2e-9, -1e-9], dtype=np.float32)

In [14]:
def quantize_tensor(tensor, scale, zero_point):
    q=np.round(tensor/scale) + zero_point
    q=np.clip(q,-128,127)
    return q.astype(np.int8)

In [15]:
def dequantize_tensor(quantized_tensor, scale, zero_point):
    return (quantized_tensor - zero_point) * scale

In [16]:
def calculate_scale_zero_point(tensor, q_min=-128, q_max=127):
    x_max=np.max(tensor)
    x_min=np.min(tensor)
    if x_max == x_min:
        scale = 1.0
        zero_point = 0
        return scale, zero_point

    scale=(x_max-x_min)/(q_max-q_min)
    zero_point=round(q_min-(x_min/scale))
    zero_point=np.clip(zero_point,q_min,q_max)
    return scale,zero_point
scale,zp=calculate_scale_zero_point(t1, q_min=-128, q_max=127)

In [17]:
tensors = {
    "Tensor 1": t1,
    "Tensor 2": t2,
    "Tensor 3": t3,
    "Tensor 4": t4,
    "Tensor 5": t5
}

for name, tensor in tensors.items():

    scale, zero_point = calculate_scale_zero_point(tensor)

    quantized_tensor = quantize_tensor(tensor, scale, zero_point)

    dequantized_tensor = dequantize_tensor(
        quantized_tensor,
        scale,
        zero_point
    )

    mae = np.mean(np.abs(tensor - dequantized_tensor))

    print("\n------------------------------")
    print(name)
    print("------------------------------")

    print("Tensor Values:")
    print(tensor)

    print("Tensor Min:", np.min(tensor))
    print("Tensor Max:", np.max(tensor))

    print("Scale:", scale)
    print("Zero Point:", zero_point)

    print("Quantized Tensor:")
    print(quantized_tensor)

    print("Dequantized Tensor:")
    print(dequantized_tensor)

    print("Mean Absolute Error:", mae)


------------------------------
Tensor 1
------------------------------
Tensor Values:
[-1.5 -0.8  0.   0.9  2.3]
Tensor Min: -1.5
Tensor Max: 2.3
Scale: 0.01490196
Zero Point: -27
Quantized Tensor:
[-128  -81  -27   33  127]
Dequantized Tensor:
[-1.50509799 -0.80470585  0.          0.89411762  2.29490188]
Mean Absolute Error: 0.004156852141022682

------------------------------
Tensor 2
------------------------------
Tensor Values:
[0.1 0.5 1.2 2.  3.5]
Tensor Min: 0.1
Tensor Max: 3.5
Scale: 0.013333334
Zero Point: -128
Quantized Tensor:
[-120  -90  -38   22  127]
Dequantized Tensor:
[0.10666667 0.50666668 1.20000003 2.00000005 3.40000008]
Mean Absolute Error: 0.02266666628420353

------------------------------
Tensor 3
------------------------------
Tensor Values:
[-3.  -2.1 -1.4 -0.6 -0.1]
Tensor Min: -3.0
Tensor Max: -0.1
Scale: 0.011372549
Zero Point: 127
Quantized Tensor:
[-128  -58    4   74  118]
Dequantized Tensor:
[-2.90000011 -2.10392165 -1.39882358 -0.60274512 -0.10235295]
